<a href="https://colab.research.google.com/github/alfredo170987/Proyecto-2-Sistemas-de-Comunicaci-n/blob/main/Notebook_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Asignación Óptima de Potencia en Redes Inalámbricas Multiusuario mediante Redes Neuronales Artificiales

En entornos reales con múltiples usuarios que comparten el mismo espectro y se interfieren entre sí, calcular la potencia óptima mediante algoritmos matemáticos clásicos es demasiado lento para ejecutarse en tiempo real (en milisegundos).

En este cuaderno entrenamos una **Red Neuronal Artificial (Perceptrón Multicapa - MLP)** para aprender a imitar la solución óptima instantáneamente a partir de las ganancias del canal (CSI).

### 1. Librerías y Generación del Escenario Multiusuario
Simulamos 3 pares de transmisor-receptor que comparten la misma frecuencia y se interfieren mutuamente. Generamos 2,000 escenarios distintos de condiciones de canal (desvanecimiento Rayleigh).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import time

np.random.seed(42)

# Configuración de la red inalámbrica
NUM_USUARIOS = 3
NUM_MUESTRAS = 2000
P_MAX = 1.0       # Potencia máxima permitida por transmisor (1 Watt)
RUIDO = 0.01      # Ruido térmico

# Generamos matrices de ganancia de canal |H|^2 de tamaño (2000, 3, 3)
# H[m, i, j] = ganancia del canal desde el transmisor j hacia el receptor i en el escenario m
H_dataset = np.random.exponential(scale=1.0, size=(NUM_MUESTRAS, NUM_USUARIOS, NUM_USUARIOS))

# Hacemos que los enlaces directos (diagonal) sean naturalmente más fuertes que las interferencias
for i in range(NUM_USUARIOS):
    H_dataset[:, i, i] *= 2.5

print(f"Dataset generado: {NUM_MUESTRAS} estados de canal para una red de {NUM_USUARIOS} usuarios.")

### 2. Cálculo de la Tasa Suma (Shannon) y Datos de Entrenamiento
Definimos la función física de capacidad total de la red (*Sum-Rate*) considerando la interferencia cruzada. Luego, buscamos la potencia óptima para cada escenario para crear las etiquetas con las que aprenderá nuestra red neuronal.

In [ ]:
def calcular_sum_rate(H, potencias):
    """Calcula la capacidad total (bps/Hz) de los 3 usuarios considerando interferencia mutua."""
    sum_rate = 0.0
    for i in range(NUM_USUARIOS):
        senal_deseada = H[i, i] * potencias[i]
        interferencia = sum(H[i, j] * potencias[j] for j in range(NUM_USUARIOS) if j != i)
        sinr = senal_deseada / (interferencia + RUIDO)
        sum_rate += np.log2(1 + sinr)
    return sum_rate

# Generación de etiquetas óptimas (potencias ideales para cada escenario)
candidatos_p = np.linspace(0.0, P_MAX, 11)
combinaciones = np.array(np.meshgrid(candidatos_p, candidatos_p, candidatos_p)).T.reshape(-1, NUM_USUARIOS)

X = [] # Entradas: Matriz de ganancias H aplanada (9 valores por escenario)
Y = [] # Salidas: Vector de potencias óptimas [P1, P2, P3]

for m in range(NUM_MUESTRAS):
    H_actual = H_dataset[m]
    tasas = [calcular_sum_rate(H_actual, p) for p in combinaciones]
    mejor_p = combinaciones[np.argmax(tasas)]

    X.append(H_actual.flatten())
    Y.append(mejor_p)

X = np.array(X)
Y = np.array(Y)

# Dividimos en datos de entrenamiento (80%) y prueba (20%)
X_train, X_test, Y_train, Y_test, H_train, H_test = train_test_split(
    X, Y, H_dataset, test_size=0.2, random_state=42
)

print("Datos listos. Ejemplo de entrada (9 ganancias):", np.round(X_train[0], 2))
print("Potencias óptimas objetivo [P1, P2, P3]:", Y_train[0])

### 3. Arquitectura y Entrenamiento de la Red Neuronal (MLP)
Construimos una Red Neuronal Artificial con 2 capas ocultas (64 y 32 neuronas con activación ReLU). La red recibe los 9 coeficientes del canal y predice instantáneamente los 3 niveles de potencia.

In [ ]:
# Definición de la Red Neuronal Artificial (Perceptrón Multicapa)
red_neuronal = MLPRegressor(
    hidden_layer_sizes=(64, 32),  # 2 capas ocultas
    activation='relu',            # Función de activación no lineal
    solver='adam',                # Optimizador estándar en Deep Learning
    max_iter=300,                 # Épocas de entrenamiento
    random_state=42
)

print("Entrenando Red Neuronal Artificial...")
red_neuronal.fit(X_train, Y_train)

# Predicción sobre escenarios nuevos nunca vistos (Test Set)
Y_pred = red_neuronal.predict(X_test)
# Aseguramos que las potencias predichas estén dentro del límite físico [0, P_MAX]
Y_pred = np.clip(Y_pred, 0.0, P_MAX)

mse = mean_squared_error(Y_test, Y_pred)
print(f"Entrenamiento completado. Error Cuadrático Medio (MSE): {mse:.4f}")

### 4. Evaluación y Gráficas Comparativas
Comparamos el rendimiento de nuestra Red Neuronal frente a la solución óptima matemática, transmisión a potencia máxima fija y asignación aleatoria.

In [ ]:
tasas_optimo = []
tasas_red_neuronal = []
tasas_max_potencia = []
tasas_aleatorio = []

for i in range(len(H_test)):
    H_i = H_test[i]
    tasas_optimo.append(calcular_sum_rate(H_i, Y_test[i]))
    tasas_red_neuronal.append(calcular_sum_rate(H_i, Y_pred[i]))
    tasas_max_potencia.append(calcular_sum_rate(H_i, [P_MAX]*NUM_USUARIOS))
    tasas_aleatorio.append(calcular_sum_rate(H_i, np.random.uniform(0, P_MAX, NUM_USUARIOS)))

# 1. Gráfica de barras: Comparación de Capacidad Promedio
metodos = ['Óptimo Teórico\n(Lento)', 'Red Neuronal\n(Tiempo Real)', 'Potencia Máxima\n(Fija)', 'Asignación\nAleatoria']
promedios = [np.mean(tasas_optimo), np.mean(tasas_red_neuronal), np.mean(tasas_max_potencia), np.mean(tasas_aleatorio)]
colores = ['#2ca02c', '#1f77b4', '#ff7f0e', '#d62728']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

barras = ax1.bar(metodos, promedios, color=colores, alpha=0.85)
ax1.set_ylabel('Capacidad Total de Red (bps/Hz)')
ax1.set_title('Desempeño en Asignación de Potencia Multiusuario')
ax1.grid(axis='y', alpha=0.3)

for barra in barras:
    yval = barra.get_height()
    ax1.text(barra.get_x() + barra.get_width()/2, yval + 0.1, f"{yval:.2f}", ha='center', fontweight='bold')

# 2. Gráfica de evolución de pérdida (Loss Curve) durante el entrenamiento
ax2.plot(red_neuronal.loss_curve_, color='purple', linewidth=2)
ax2.set_xlabel('Iteraciones (Épocas)')
ax2.set_ylabel('Pérdida (Loss)')
ax2.set_title('Curva de Aprendizaje de la Red Neuronal')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

eficiencia_relativa = (np.mean(tasas_red_neuronal) / np.mean(tasas_optimo)) * 100
print(f"La Red Neuronal alcanza el {eficiencia_relativa:.1f}% del rendimiento óptimo teórico en microsegundos.")

### 5. Comparación de Latencia (Tiempo Real vs. Optimización Clásica)
En estándares como 5G y O-RAN, las decisiones de radio deben tomarse en menos de 1 milisegundo (TTI - *Transmission Time Interval*). Aquí medimos el tiempo exacto de inferencia de la Red Neuronal frente a la búsqueda iterativa clásica.

In [ ]:
# 1. Medir tiempo del Algoritmo de Optimización Tradicional (Búsqueda Exhaustiva)
inicio_trad = time.perf_counter()
for m in range(len(H_test)):
    H_actual = H_test[m]
    tasas = [calcular_sum_rate(H_actual, p) for p in combinaciones]
    _ = combinaciones[np.argmax(tasas)]
tiempo_total_trad = (time.perf_counter() - inicio_trad) * 1000  # Convertir a milisegundos
tiempo_por_escenario_trad = tiempo_total_trad / len(H_test)

# 2. Medir tiempo de Inferencia de la Red Neuronal Artificial
inicio_nn = time.perf_counter()
_ = red_neuronal.predict(X_test)
tiempo_total_nn = (time.perf_counter() - inicio_nn) * 1000  # Convertir a milisegundos
tiempo_por_escenario_nn = tiempo_total_nn / len(H_test)

# Cálculo de aceleración (Speedup)
speedup = tiempo_por_escenario_trad / tiempo_por_escenario_nn

print(f"⏱️ Tiempo promedio por decisión (Optimización Clásica): {tiempo_por_escenario_trad:.4f} ms")
print(f"⚡ Tiempo promedio por decisión (Red Neuronal):         {tiempo_por_escenario_nn:.6f} ms")
print(f"🚀 La Red Neuronal es {speedup:,.0f} veces más rápida.")

# 3. Gráfica de Comparación de Latencia (Escala Logarítmica)
fig, ax = plt.subplots(figsize=(8, 5))

metodos_tiempo = ['Optimización Clásica\n(Iterativa)', 'Red Neuronal Artificial\n(Inferencia)']
tiempos = [tiempo_por_escenario_trad, tiempo_por_escenario_nn]
colores_tiempo = ['#d62728', '#2ca02c']

barras = ax.bar(metodos_tiempo, tiempos, color=colores_tiempo, width=0.5, alpha=0.85)
ax.set_yscale('log')  # Escala logarítmica para visualizar diferencias de varios órdenes de magnitud
ax.set_ylabel('Tiempo por Decisión (ms) - Escala Log')
ax.set_title('Latencia de Toma de Decisiones: Viabilidad en Tiempo Real (5G/6G)')

# Línea de referencia del estándar 5G (1 ms por ranura de tiempo)
ax.axhline(y=1.0, color='blue', linestyle='--', linewidth=1.5, label='Límite Tiempo Real 5G (1 ms)')
ax.legend()
ax.grid(True, which="both", ls="-", alpha=0.2)

for barra in barras:
    yval = barra.get_height()
    ax.text(barra.get_x() + barra.get_width()/2, yval * 1.3, f"{yval:.4f} ms", ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

### Conclusiones Clave para la Presentación
1. **Mitigación inteligente de interferencia:** Transmitir siempre a potencia máxima degrada la capacidad global de la red debido a la interferencia mutua. La Red Neuronal aprende a reducir o apagar la potencia de usuarios con enlaces desfavorables para maximizar el rendimiento conjunto.
2. **Viabilidad en tiempo real (5G/O-RAN):** Mientras que la búsqueda matemática iterativa supera el umbral de 1 ms exigido por el estándar celular, la inferencia de la Red Neuronal toma decisiones en fracciones de microsegundo.
3. **Escalabilidad:** El entrenamiento pesado ocurre fuera de línea (*offline*), permitiendo que el modelo desplegado en la estación base opere con muy bajo costo computacional.